In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
 
CONFIG = {
    "catalog": "workspace",
    "schema":  "banking_datasentry",
}
 
REMEDIATED_AT = datetime.now().isoformat()
 
spark.sql(f"USE CATALOG {CONFIG['catalog']}")
spark.sql(f"USE SCHEMA {CONFIG['schema']}")
 
# Rule classification — how each failed rule is handled
RULE_CLASSIFICATION = {
    # AUTO-FIX — fix programmatically
    "customer_id_unique":                "AUTO_FIX",
    "account_id_unique":                 "AUTO_FIX",
    "transaction_id_unique":             "AUTO_FIX",
    "country_code_valid":                "AUTO_FIX",
 
    # QUARANTINE — needs source system fix
    "currency_valid":                    "QUARANTINE",
    "amount_greater_than_zero":          "QUARANTINE",
    "transaction_date_not_future":       "QUARANTINE",
    "opened_date_not_future":            "QUARANTINE",
    "savings_loan_balance_not_negative": "QUARANTINE",
    "transaction_type_valid":            "QUARANTINE",
    "account_type_valid":                "QUARANTINE",
    "status_valid":                      "QUARANTINE",
    "balance_valid":                     "QUARANTINE",
    "amount_valid":                      "QUARANTINE",
 
    # ESCALATE — needs human review
    "customer_id_not_null":              "ESCALATE",
    "full_name_not_null":                "ESCALATE",
    "email_not_null":                    "ESCALATE",
    "created_at_not_null":               "ESCALATE",
    "account_id_not_null":               "ESCALATE",
    "transaction_id_not_null":           "ESCALATE",
    "amount_not_null":                   "ESCALATE",
    "email_format_valid":                "ESCALATE",
    "customer_id_exists_in_customers":   "ESCALATE",
    "account_id_exists_in_accounts":     "ESCALATE",
    "failed_reversed_amount_not_positive": "ESCALATE",
    "customer_id_uuid_format":           "ESCALATE",
    "account_id_uuid_format":            "ESCALATE",
    "transaction_id_uuid_format":        "ESCALATE",
    "age_between_18_and_120":            "ESCALATE",
    "created_at_not_future":             "ESCALATE",
    "date_of_birth_not_future":          "ESCALATE",
    "date_of_birth_valid":               "ESCALATE",
    "opened_date_valid":                 "ESCALATE",
}
 
# Country code standardisation map
COUNTRY_STANDARDISATION = {
    "IRELAND": "IE", "IRE": "IE", "IRL": "IE",
    "GBR": "GB", "BRITAIN": "GB", "UK": "GB",
    "USA": "US", "GERMANY": "DE", "FRANCE": "FR", "INDIA": "IN",
}
 
print(f"Catalog       : {CONFIG['catalog']}")
print(f"Schema        : {CONFIG['schema']}")
print(f"Remediated at : {REMEDIATED_AT}")

Catalog       : workspace
Schema        : banking_datasentry
Remediated at : 2026-06-20T19:59:12.624751


In [0]:
def classify_record(failed_rules_str):
    if not failed_rules_str:
        return "ESCALATE"
    rules = [r.strip() for r in failed_rules_str.split(",")]
    classifications = set()
    for rule in rules:
        classifications.add(RULE_CLASSIFICATION.get(rule, "ESCALATE"))
    if "ESCALATE" in classifications:
        return "ESCALATE"
    if "QUARANTINE" in classifications:
        return "QUARANTINE"
    return "AUTO_FIX"
 
classify_udf = F.udf(classify_record, T.StringType())
print("classify_udf registered.")

classify_udf registered.


In [0]:
def apply_deduplication(df, pk_col):
    """Keep first occurrence of each primary key. Drop duplicates."""
    from pyspark.sql.window import Window
    window  = Window.partitionBy(pk_col).orderBy(F.col("ingested_at").asc())
    df      = df.withColumn("_rank", F.row_number().over(window))
    deduped = df.filter(F.col("_rank") == 1).drop("_rank")
    dropped = df.filter(F.col("_rank") > 1).drop("_rank")
    print(f"  Deduplication: kept {deduped.count():,} | dropped {dropped.count():,} duplicates")
    return deduped
 
def apply_country_standardisation(df):
    """Standardise known country code variants to ISO 2-letter codes."""
    mapping_expr = F.create_map(*[
        item for pair in [(F.lit(k), F.lit(v))
        for k, v in COUNTRY_STANDARDISATION.items()]
        for item in pair
    ])
    df = df.withColumn(
        "country",
        F.coalesce(mapping_expr[F.col("country")], F.col("country"))
    )
    print(f"  Country standardisation applied.")
    return df
 
print("Auto-fix helpers defined.")

Auto-fix helpers defined.


In [0]:
RULE_DIMENSION_MAP = {
    "customer_id_not_null": "Completeness", "full_name_not_null": "Completeness",
    "email_not_null": "Completeness", "created_at_not_null": "Completeness",
    "email_format_valid": "Validity", "date_of_birth_valid": "Validity",
    "country_code_valid": "Validity", "customer_id_uuid_format": "Validity",
    "created_at_not_future": "Consistency", "date_of_birth_not_future": "Consistency",
    "age_between_18_and_120": "Consistency", "customer_id_unique": "Uniqueness",
    "account_id_not_null": "Completeness", "account_type_not_null": "Completeness",
    "status_not_null": "Completeness", "account_type_valid": "Validity",
    "status_valid": "Validity", "currency_valid": "Validity",
    "account_id_uuid_format": "Validity", "opened_date_valid": "Validity",
    "savings_loan_balance_not_negative": "Consistency", "opened_date_not_future": "Consistency",
    "account_id_unique": "Uniqueness", "customer_id_exists_in_customers": "Referential Integrity",
    "transaction_id_not_null": "Completeness", "transaction_type_not_null": "Completeness",
    "amount_not_null": "Completeness", "currency_not_null": "Completeness",
    "transaction_date_not_null": "Completeness", "transaction_type_valid": "Validity",
    "transaction_id_uuid_format": "Validity", "amount_greater_than_zero": "Validity",
    "transaction_date_not_future": "Consistency", "failed_reversed_amount_not_positive": "Consistency",
    "transaction_id_unique": "Uniqueness", "account_id_exists_in_accounts": "Referential Integrity",
    "balance_valid": "Validity", "amount_valid": "Validity",
}

def get_dimensions(failed_rules_str):
    if not failed_rules_str:
        return ""
    dims = set()
    for rule in failed_rules_str.split(","):
        rule = rule.strip()
        dims.add(RULE_DIMENSION_MAP.get(rule, "Unknown"))
    return ", ".join(sorted(dims))

dimensions_udf = F.udf(get_dimensions, T.StringType())
print("dimensions_udf registered.")

def get_priority(failed_dimensions_str):
    """Derive priority from failed dimensions — same logic as severity in Notebook 05."""
    if not failed_dimensions_str:
        return "LOW"
    dims = failed_dimensions_str.split(", ")
    if "Referential Integrity" in dims or "Uniqueness" in dims:
        return "HIGH"
    if "Validity" in dims or "Consistency" in dims:
        return "MEDIUM"
    return "LOW"

priority_udf = F.udf(get_priority, T.StringType())
print("priority_udf registered.")

dimensions_udf registered.
priority_udf registered.


In [0]:
def process_table(silver_table, pk_col, table_name, apply_country=False):
    """
    Read FAIL records from a Silver table.
    Classify into AUTO_FIX / QUARANTINE / ESCALATE.
    Apply auto-fixes, re-check them, and promote only records that now pass.
    Return fixed, quarantine, escalate DataFrames and before/after stats.
    """
    silver_df   = spark.read.format("delta").table(silver_table)
    total       = silver_df.count()
    pass_before = silver_df.filter(F.col("dq_status") == "PASS").count()
    fail_df     = silver_df.filter(F.col("dq_status") == "FAIL")
    fail_count  = fail_df.count()

    print(f"\n[{table_name.upper()}]")
    print(f"  Total          : {total:,}")
    print(f"  PASS before    : {pass_before:,} ({round(pass_before/total*100,2)}%)")
    print(f"  FAIL before    : {fail_count:,}  ({round(fail_count/total*100,2)}%)")

    # Classify each FAIL record
    fail_df = fail_df.withColumn(
        "remediation_action", classify_udf(F.col("failed_rules"))
    )

    auto_fix_df   = fail_df.filter(F.col("remediation_action") == "AUTO_FIX")
    quarantine_df = fail_df.filter(F.col("remediation_action") == "QUARANTINE")
    escalate_df   = fail_df.filter(F.col("remediation_action") == "ESCALATE")

    print(f"  AUTO_FIX       : {auto_fix_df.count():,}")
    print(f"  QUARANTINE     : {quarantine_df.count():,}")
    print(f"  ESCALATE       : {escalate_df.count():,}")

    # Apply auto-fixes
    fixed_df = apply_deduplication(auto_fix_df, pk_col)
    dupes_dropped = auto_fix_df.count() - fixed_df.count()

    if apply_country:
        fixed_df = apply_country_standardisation(fixed_df)

        # FIX 2 — Re-check after the fix.
        # Only promote rows whose country is now a valid 2-letter code.
        # Unmapped values (e.g. "Ireland", "IRISH") go to escalation instead.
        valid_country = F.col("country").isNull() | F.col("country").rlike("^[A-Z]{2}$")
        still_bad = fixed_df.filter(~valid_country) \
                            .withColumn("remediation_action", F.lit("ESCALATE"))
        fixed_df  = fixed_df.filter(valid_country)
        escalate_df = escalate_df.unionByName(still_bad)
        print(f"  Country fix failed, sent to escalation: {still_bad.count():,}")

    # Tag fixed records as PASS
    fixed_df = fixed_df \
        .withColumn("dq_status",          F.lit("PASS")) \
        .withColumn("failed_rules",       F.lit("")) \
        .withColumn("remediation_action", F.lit("AUTO_FIX")) \
        .withColumn("remediated_at",      F.lit(REMEDIATED_AT))

    # Compute before/after stats
    fixed_count   = fixed_df.count()
    pass_after    = pass_before + fixed_count
    rate_before   = round(pass_before / total * 100, 2)
    effective_total = total - dupes_dropped
    rate_after    = round(pass_after / effective_total * 100, 2)
    improvement   = round(rate_after - rate_before, 2)

    print(f"  Pass rate BEFORE : {rate_before}%")
    print(f"  Pass rate AFTER  : {rate_after}%")
    print(f"  Improvement      : +{improvement}%")

    stats = {
        "table_name":       table_name,
        "total_records":    total,
        "fail_before":      fail_count,
        "pass_rate_before": rate_before,
        "auto_fixed":       fixed_count,
        "quarantined":      quarantine_df.count(),
        "escalated":        escalate_df.count(),
        "duplicates_dropped": dupes_dropped,
        "pass_rate_after":  rate_after,
        "improvement_pct":  improvement,
        "remediated_at":    REMEDIATED_AT,
    }

    return fixed_df, quarantine_df, escalate_df, stats

In [0]:
print("=" * 60)
print("REMEDIATION — START")
print("=" * 60)
 
cust_fixed, cust_quar, cust_esc, cust_stats = process_table(
    "silver_customers", "customer_id", "customers", apply_country=True
)
 
acc_fixed, acc_quar, acc_esc, acc_stats = process_table(
    "silver_accounts", "account_id", "accounts"
)
 
txn_fixed, txn_quar, txn_esc, txn_stats = process_table(
    "silver_transactions", "transaction_id", "transactions"
)

REMEDIATION — START

[CUSTOMERS]
  Total          : 55,900
  PASS before    : 51,814 (92.69%)
  FAIL before    : 4,086  (7.31%)
  AUTO_FIX       : 1,765
  QUARANTINE     : 0
  ESCALATE       : 2,321
  Deduplication: kept 977 | dropped 788 duplicates
  Country standardisation applied.
  Pass rate BEFORE : 92.69%
  Pass rate AFTER  : 94.44%
  Improvement      : +1.75%

[ACCOUNTS]
  Total          : 60,000
  PASS before    : 53,068 (88.45%)
  FAIL before    : 6,932  (11.55%)
  AUTO_FIX       : 0
  QUARANTINE     : 2,179
  ESCALATE       : 4,753
  Deduplication: kept 0 | dropped 0 duplicates
  Pass rate BEFORE : 88.45%
  Pass rate AFTER  : 88.45%
  Improvement      : +0.0%

[TRANSACTIONS]
  Total          : 207,400
  PASS before    : 160,327 (77.3%)
  FAIL before    : 47,073  (22.7%)
  AUTO_FIX       : 3,249
  QUARANTINE     : 6,071
  ESCALATE       : 37,753
  Deduplication: kept 1,622 | dropped 1,627 duplicates
  Pass rate BEFORE : 77.3%
  Pass rate AFTER  : 78.09%
  Improvement      : +0

In [0]:
# overwriteSchema allows the new rule_balance_valid / rule_amount_valid
# columns (added in Notebook 03) to flow through to the remediated tables
cust_fixed.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_remediated_customers")
acc_fixed.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_remediated_accounts")
txn_fixed.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_remediated_transactions")

print("silver_remediated tables written:")
print(f"  silver_remediated_customers    : {cust_fixed.count():,} rows")
print(f"  silver_remediated_accounts     : {acc_fixed.count():,} rows")
print(f"  silver_remediated_transactions : {txn_fixed.count():,} rows")

silver_remediated tables written:
  silver_remediated_customers    : 977 rows
  silver_remediated_accounts     : 0 rows
  silver_remediated_transactions : 1,622 rows


In [0]:
def build_remediation_log(fixed_df, table_name, pk_col):
    return fixed_df.select(
        F.expr("uuid()").alias("log_id"),
        F.lit(table_name).alias("table_name"),
        F.col(pk_col).alias("record_id"),
        F.lit("AUTO_FIX").alias("remediation_type"),
        F.lit("PASS").alias("new_status"),
        F.lit(REMEDIATED_AT).alias("remediated_at"),
    )
 
gold_remediation_log = \
    build_remediation_log(cust_fixed, "customers",    "customer_id") \
    .union(build_remediation_log(acc_fixed,  "accounts",     "account_id")) \
    .union(build_remediation_log(txn_fixed,  "transactions",  "transaction_id"))
 
gold_remediation_log.write.format("delta").mode("overwrite").saveAsTable("gold_remediation_log")
print(f"gold_remediation_log written: {gold_remediation_log.count():,} rows")

gold_remediation_log written: 2,599 rows


In [0]:
cust_quar = cust_quar.withColumn("failed_dimensions", dimensions_udf(F.col("failed_rules")))
acc_quar  = acc_quar.withColumn("failed_dimensions", dimensions_udf(F.col("failed_rules")))
txn_quar  = txn_quar.withColumn("failed_dimensions", dimensions_udf(F.col("failed_rules")))

cust_esc = cust_esc.withColumn("failed_dimensions", dimensions_udf(F.col("failed_rules")))
acc_esc  = acc_esc.withColumn("failed_dimensions", dimensions_udf(F.col("failed_rules")))
txn_esc  = txn_esc.withColumn("failed_dimensions", dimensions_udf(F.col("failed_rules")))

def build_quarantine_log(df, table_name, pk_col):
    return df.select(
        F.expr("uuid()").alias("log_id"),
        F.lit(table_name).alias("table_name"),
        F.col(pk_col).alias("record_id"),
        F.col("failed_rules"),
        F.col("failed_dimensions"),
        F.lit("PENDING").alias("status"),
        F.lit("Needs source system fix").alias("reason"),
        F.lit(REMEDIATED_AT).alias("quarantined_at"),
    )
 
gold_quarantine_log = \
    build_quarantine_log(cust_quar, "customers",    "customer_id") \
    .union(build_quarantine_log(acc_quar,  "accounts",     "account_id")) \
    .union(build_quarantine_log(txn_quar,  "transactions",  "transaction_id"))
 
gold_quarantine_log.write.format("delta").mode("overwrite").saveAsTable("gold_quarantine_log")
print(f"gold_quarantine_log written: {gold_quarantine_log.count():,} rows")

gold_quarantine_log written: 8,250 rows


In [0]:
def build_escalation_log(df, table_name, pk_col):
    df = df.withColumn("priority", priority_udf(F.col("failed_dimensions")))
    return df.select(
        F.expr("uuid()").alias("log_id"),
        F.lit(table_name).alias("table_name"),
        F.col(pk_col).alias("record_id"),
        F.col("failed_rules"),
        F.col("failed_dimensions"),
        F.col("priority"),
        F.lit("OPEN").alias("status"),
        F.lit("Needs human review").alias("escalation_reason"),
        F.lit(REMEDIATED_AT).alias("escalated_at"),
    )
 
gold_escalation_log = \
    build_escalation_log(cust_esc, "customers",    "customer_id") \
    .union(build_escalation_log(acc_esc,  "accounts",     "account_id")) \
    .union(build_escalation_log(txn_esc,  "transactions",  "transaction_id"))
 
gold_escalation_log.write.format("delta").mode("overwrite").saveAsTable("gold_escalation_log")
print(f"gold_escalation_log written: {gold_escalation_log.count():,} rows")

gold_escalation_log written: 44,827 rows


In [0]:
gold_remediation_summary = spark.createDataFrame([
    cust_stats, acc_stats, txn_stats
])

# overwriteSchema allows the new duplicates_dropped column to be added
gold_remediation_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_remediation_summary")

print(f"gold_remediation_summary written: {gold_remediation_summary.count()} rows")

gold_remediation_summary written: 3 rows


In [0]:
print("=" * 60)
print("DATASENTRY PHASE 2 — REMEDIATION SUMMARY")
print("=" * 60)

all_stats      = [cust_stats, acc_stats, txn_stats]
total_records  = sum(s["total_records"] for s in all_stats)
total_fail     = sum(s["fail_before"]   for s in all_stats)
total_fixed    = sum(s["auto_fixed"]    for s in all_stats)
total_quar     = sum(s["quarantined"]   for s in all_stats)
total_esc      = sum(s["escalated"]     for s in all_stats)
total_dupes    = sum(s.get("duplicates_dropped", 0) for s in all_stats)

# Dropped duplicates no longer exist in the cleaned dataset,
# so they come out of the denominator for the AFTER rate —
# same logic as the per-table rates in process_table()
overall_before = round((total_records - total_fail) / total_records * 100, 2)
overall_after  = round((total_records - total_fail + total_fixed) /
                       (total_records - total_dupes) * 100, 2)
improvement    = round(overall_after - overall_before, 2)

print(f"\nOverall Pipeline:")
print(f"  Pass rate BEFORE remediation : {overall_before}%")
print(f"  Pass rate AFTER  remediation : {overall_after}%")
print(f"  Improvement                  : +{improvement}%")
print(f"  Auto-fixed                   : {total_fixed:,}")
print(f"  Duplicates dropped           : {total_dupes:,}")
print(f"  Quarantined                  : {total_quar:,}")
print(f"  Escalated                    : {total_esc:,}")

print(f"\n{'TABLE':<15} {'BEFORE':>8} {'FIXED':>8} {'DUPES':>8} {'QUAR':>8} {'ESC':>8} {'AFTER':>8} {'GAIN':>8}")
print("-" * 81)
for s in all_stats:
    print(f"{s['table_name']:<15} {s['pass_rate_before']:>7}% "
          f"{s['auto_fixed']:>8,} {s.get('duplicates_dropped', 0):>8,} "
          f"{s['quarantined']:>8,} {s['escalated']:>8,} "
          f"{s['pass_rate_after']:>7}% {s['improvement_pct']:>+7}%")

print("\nNotebook 06 complete.")
print("Gold scoring (Notebook 04) shows the detection view of full Silver.")
print("The before/after view is in gold_remediation_summary.")

DATASENTRY PHASE 2 — REMEDIATION SUMMARY

Overall Pipeline:
  Pass rate BEFORE remediation : 82.03%
  Pass rate AFTER  remediation : 82.84%
  Improvement                  : +0.81%
  Auto-fixed                   : 2,599
  Quarantined                  : 8,250
  Escalated                    : 44,827

TABLE             BEFORE    FIXED     QUAR      ESC    AFTER     GAIN
------------------------------------------------------------------------
customers         92.69%      977        0    2,321   94.44%   +1.75%
accounts          88.45%        0    2,179    4,753   88.45%    +0.0%
transactions       77.3%    1,622    6,071   37,753   78.09%   +0.79%

Notebook 06 complete.
Next: Notebook 04 Gold Scoring reads PASS + auto-fixed records.
